In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [14]:
def gerar_grafico_artigo(caminho_csv, coluna_index, nome_arquivo_saida="grafico_artigo.png"):
    """
    Gera um gráfico em painel para artigos científicos de forma 100% dinâmica.
    Funciona para qualquer quantidade de categorias/linhas no CSV.
    
    Parâmetros:
    - caminho_csv (str): Caminho para o arquivo .csv
    - coluna_index (str): Nome da coluna que servirá de eixo X ('filtroApi', 'modeloClassificador', etc.)
    - nome_arquivo_saida (str): Nome do arquivo .png que será salvo.
    """
    # 1. Carga e preparação dinâmica dos dados
    df = pd.read_csv(caminho_csv)
    df[coluna_index] = df[coluna_index].astype(str)
    
    x_labels = df[coluna_index].tolist()
    num_categorias = len(df)
    x = np.arange(num_categorias)
    largura_barra = 0.35
    offset = largura_barra / 2
    cores = ['#2ca02c', '#ff7f0e', '#d62728'] # Verde, Laranja, Vermelho
    
    acertos_completos = df['acertosCompletosPorcentagem'].tolist()
    acertos_top5 = df['acertosTop5Porcentagem'].tolist()
    erros = df['errosPorcentagem'].tolist()
    tempos = df['tempo_execucao'].tolist()
    
    # Matriz auxiliar para as bases de empilhamento
    bottom_erros = [i + j for i, j in zip(acertos_completos, acertos_top5)]
    
    # Configuração da figura
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={'width_ratios': [1.5, 1]})
    
    # -----------------------------------------------------------------
    # GRÁFICO 1: Barras Empilhadas Dinâmicas
    # -----------------------------------------------------------------
    ax1.bar(x, acertos_completos, label='Acertos Completos', color=cores[0], width=largura_barra, zorder=3)
    ax1.bar(x, acertos_top5, bottom=acertos_completos, label='Acertos Top 5', color=cores[1], width=largura_barra, zorder=3)
    ax1.bar(x, erros, bottom=bottom_erros, label='Erros', color=cores[2], width=largura_barra, zorder=3)
    
    # Textos internos das porcentagens em cada barra existente
    for i in range(num_categorias):
        ax1.text(i, acertos_completos[i]/2, f"{acertos_completos[i]:.1f}%", ha='center', va='center', color='white', weight='bold', fontsize=9, zorder=4)
        ax1.text(i, acertos_completos[i] + acertos_top5[i]/2, f"{acertos_top5[i]:.1f}%", ha='center', va='center', color='black', fontsize=8, zorder=4)
        ax1.text(i, bottom_erros[i] + erros[i]/2, f"{erros[i]:.1f}%", ha='center', va='center', color='white', weight='bold', fontsize=9, zorder=4)
    
    # Conexões e Variações Sequenciais (Par a Par: 0->1, 1->2...)
    for idx in range(num_categorias - 1):
        # Cálculos de variação para o par atual
        v_verde = ((acertos_completos[idx+1] - acertos_completos[idx]) / acertos_completos[idx]) * 100 if acertos_completos[idx] > 0 else 0
        v_laranja = ((acertos_top5[idx+1] - acertos_top5[idx]) / acertos_top5[idx]) * 100 if acertos_top5[idx] > 0 else 0
        v_vermelho = ((erros[idx+1] - erros[idx]) / erros[idx]) * 100 if erros[idx] > 0 else 0
        v_pares = [v_verde, v_laranja, v_vermelho]
        
        for i in range(3):
            if i == 0:
                yf_top, yt_top = acertos_completos[idx], acertos_completos[idx+1]
                yf_mid, yt_mid = yf_top / 2, yt_top / 2
            elif i == 1:
                yf_top, yt_top = bottom_erros[idx], bottom_erros[idx+1]
                yf_mid = acertos_completos[idx] + acertos_top5[idx] / 2
                yt_mid = acertos_completos[idx+1] + acertos_top5[idx+1] / 2
            else:
                yf_top, yt_top = 100, 100
                yf_mid = bottom_erros[idx] + erros[idx] / 2
                yt_mid = bottom_erros[idx+1] + erros[idx+1] / 2
    
            # Linha tracejada conectando a barra atual à próxima
            ax1.plot([idx + offset, (idx + 1) - offset], [yf_top, yt_top], color=cores[i], linestyle='--', linewidth=1.1, zorder=2)
            
            # Caixa de texto centralizada entre o par
            sinal = "+" if v_pares[i] > 0 else ""
            texto_var = f"{sinal}{v_pares[i]:.0f}%"
            x_centro = idx + 0.5
            y_centro = (yf_mid + yt_mid) / 2
            
            # Só plota a variação se ela for numericamente relevante para evitar poluição visual
            if abs(v_pares[i]) >= 0.5:
                ax1.text(x_centro, y_centro, texto_var, color=cores[i], weight='bold', fontsize=8.5,
                         ha='center', va='center', zorder=4,
                         bbox=dict(facecolor='white', edgecolor=cores[i], boxstyle='round,pad=0.2', alpha=0.9))
                
    # Customização do Eixo 1
    ax1.set_xticks(x)
    ax1.set_xticklabels(x_labels, fontsize=10, weight='bold', rotation=15 if num_categorias > 2 else 0)
    ax1.set_xlabel(coluna_index, fontsize=12)
    ax1.set_ylabel('Composição dos Resultados (%)', fontsize=12)
    ax1.set_ylim(0, 105)
    ax1.grid(axis='y', linestyle=':', alpha=0.6, zorder=0)
    ax1.legend(loc='lower left', fontsize=9)
    
    # -----------------------------------------------------------------
    # GRÁFICO 2: Tempo de Execução Dinâmico (Anti-viés: começando em 0)
    # -----------------------------------------------------------------
    # Mapeia uma paleta de azuis baseada na quantidade de barras que existirem
    tons_azul = plt.cm.Blues(np.linspace(0.4, 0.7, num_categorias))
    bars_time = ax2.bar(x, tempos, color=tons_azul, width=largura_barra, edgecolor='black', linewidth=0.7, zorder=3)
    
    # Rótulo com o tempo real no topo de cada barra
    for bar in bars_time:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + (max(tempos) * 0.015), f"{height:.1f}s", 
                 ha='center', va='bottom', weight='bold', fontsize=9, zorder=4)
        
    ax2.set_ylim(0, max(tempos) * 1.3) # Espaço para as linhas de cota superiores
    
    # Cria as linhas de variação de tempo par a par de forma dinâmica
    for idx in range(num_categorias - 1):
        t_atual = tempos[idx]
        t_proximo = tempos[idx+1]
        d_s = t_proximo - t_atual
        d_pct = (d_s / t_atual) * 100
        
        # Define uma altura incremental para as cotas não se sobreporem se houverem muitas barras
        y_linha_cota = max(tempos) + (max(tempos) * 0.08 * (idx + 1))
        
        # Desenha a linha de cota estilo "grampo"
        ax2.plot([idx, idx, idx+1, idx+1], [t_atual + (max(tempos)*0.03), y_linha_cota, y_linha_cota, t_proximo + (max(tempos)*0.03)], 
                 color='gray', linewidth=1.0, linestyle='-', zorder=2)
        
        sinal_t = "+" if d_s > 0 else ""
        texto_tempo = f"Δ: {d_s:.1f}s ({sinal_t}{d_pct:.1f}%)"
        
        ax2.text(idx + 0.5, y_linha_cota, texto_tempo, color='black', weight='bold', fontsize=8,
                 ha='center', va='center', zorder=4,
                 bbox=dict(facecolor='#f9f9f9', edgecolor='gray', boxstyle='square,pad=0.3'))
        
    # Customização do Eixo 2
    ax2.set_xticks(x)
    ax2.set_xticklabels(x_labels, fontsize=10, weight='bold', rotation=15 if num_categorias > 2 else 0)
    ax2.set_xlabel(coluna_index, fontsize=12)
    ax2.set_ylabel('Tempo de Execução (segundos)', fontsize=12)
    ax2.grid(axis='y', linestyle=':', alpha=0.6, zorder=0)
    
    # Salvamento de alta qualidade
    plt.tight_layout()
    plt.savefig(nome_arquivo_saida, dpi=300)
    plt.close()
    print(f"Sucesso: '{nome_arquivo_saida}' gerado com base em '{coluna_index}'.")

Embedding

In [15]:
gerar_grafico_artigo('embedding.csv', 'embedding', 'analise_embedding.png')

Sucesso: 'analise_embedding.png' gerado com base em 'embedding'.


FiltroAPI

In [16]:
gerar_grafico_artigo('filtroApi.csv', 'filtroApi', 'analise_filtroApi.png')

Sucesso: 'analise_filtroApi.png' gerado com base em 'filtroApi'.


listaFiltroApis

In [17]:
gerar_grafico_artigo('listaFiltroApis.csv', 'listaFiltroApis', 'analise_listaFiltroApis.png')

Sucesso: 'analise_listaFiltroApis.png' gerado com base em 'listaFiltroApis'.


modeloClassificador

In [18]:
gerar_grafico_artigo('modeloClassificador.csv', 'modeloClassificador', 'analise_modeloClassificador.png')

Sucesso: 'analise_modeloClassificador.png' gerado com base em 'modeloClassificador'.


rerank

In [19]:
gerar_grafico_artigo('rerank.csv', 'rerank', 'analise_rerank.png')

Sucesso: 'analise_rerank.png' gerado com base em 'rerank'.


melhores

In [25]:
def gerar_grafico_horizontal(caminho_csv, nome_arquivo_saida="resultado_horizontal.png"):
    """
    Gera um gráfico com barras horizontais empilhadas em duas linhas (painel vertical),
    facilitando a leitura de múltiplas configurações de hiperparâmetros para artigos.
    """
    # 1. Ler e ordenar o CSV (da maior taxa de acerto para a menor)
    df = pd.read_csv(caminho_csv)
    df = df.sort_values(by='acertosCompletosPorcentagem', ascending=True).reset_index(drop=True)
    
    num_configuracoes = len(df)
    df['ID_Config'] = [f"C{num_configuracoes - i}" for i in range(num_configuracoes)]
    
    y_labels = df['ID_Config'].tolist()
    y = np.arange(num_configuracoes)
    altura_barra = 0.45
    offset = altura_barra / 2
    cores = ['#2ca02c', '#ff7f0e', '#d62728'] # Verde, Laranja, Vermelho
    
    acertos_completos = df['acertosCompletosPorcentagem'].tolist()
    acertos_top5 = df['acertosTop5Porcentagem'].tolist()
    erros = df['errosPorcentagem'].tolist()
    tempos = df['tempo_execucao'].tolist()
    
    bottom_erros = [i + j for i, j in zip(acertos_completos, acertos_top5)]
    
    # Configuração da figura (2 linhas de gráficos, 1 coluna)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 11), sharey=True)
    
    # -----------------------------------------------------------------
    # GRÁFICO 1 (Topo): Composição de Acertos/Erros Horizontal
    # -----------------------------------------------------------------
    ax1.barh(y, acertos_completos, label='Acertos Completos', color=cores[0], height=altura_barra, zorder=3)
    ax1.barh(y, acertos_top5, left=acertos_completos, label='Acertos Top 5', color=cores[1], height=altura_barra, zorder=3)
    ax1.barh(y, erros, left=bottom_erros, label='Erros', color=cores[2], height=altura_barra, zorder=3)
    
    # Textos internos das porcentagens dentro das barras
    for i in range(num_configuracoes):
        ax1.text(acertos_completos[i]/2, i, f"{acertos_completos[i]:.1f}%", ha='center', va='center', color='white', weight='bold', fontsize=8.5, zorder=4)
        if acertos_top5[i] > 1.5:
            ax1.text(acertos_completos[i] + acertos_top5[i]/2, i, f"{acertos_top5[i]:.1f}%", ha='center', va='center', color='black', fontsize=8, zorder=4)
        ax1.text(bottom_erros[i] + erros[i]/2, i, f"{erros[i]:.1f}%", ha='center', va='center', color='white', weight='bold', fontsize=8.5, zorder=4)
        
    # Conexões e Variações Verticais (Entre as barras vizinhas)
    for idx in range(num_configuracoes - 1):
        v_verde = ((acertos_completos[idx+1] - acertos_completos[idx]) / acertos_completos[idx]) * 100
        v_laranja = ((acertos_top5[idx+1] - acertos_top5[idx]) / acertos_top5[idx]) * 100 if acertos_top5[idx] > 0 else 0
        v_vermelho = ((erros[idx+1] - erros[idx]) / erros[idx]) * 100
        v_pares = [v_verde, v_laranja, v_vermelho]
        
        for i in range(3):
            if i == 0:
                xf_top, xt_top = acertos_completos[idx], acertos_completos[idx+1]
                xf_mid, xt_mid = xf_top / 2, xt_top / 2
            elif i == 1:
                xf_top, xt_top = bottom_erros[idx], bottom_erros[idx+1]
                xf_mid = acertos_completos[idx] + acertos_top5[idx] / 2
                xt_mid = acertos_completos[idx+1] + acertos_top5[idx+1] / 2
            else:
                xf_top, xt_top = 100, 100
                xf_mid = bottom_erros[idx] + erros[idx] / 2
                xt_mid = bottom_erros[idx+1] + erros[idx+1] / 2

            # Linha tracejada vertical conectando a base da barra de baixo ao topo da de cima
            ax1.plot([xf_top, xt_top], [idx + offset, (idx + 1) - offset], color=cores[i], linestyle='--', linewidth=1.0, alpha=0.7, zorder=2)
            
            # Caixa de texto com a variação entre as configurações
            if abs(v_pares[i]) >= 0.5:
                sinal = "+" if v_pares[i] > 0 else ""
                x_centro = (xf_mid + xt_mid) / 2
                ax1.text(x_centro, idx + 0.5, f"{sinal}{v_pares[i]:.0f}%", color=cores[i], weight='bold', fontsize=8,
                         ha='center', va='center', zorder=4,
                         bbox=dict(facecolor='white', edgecolor=cores[i], boxstyle='round,pad=0.15', alpha=0.9))

    # Customização do Gráfico 1 (Composição)
    ax1.set_yticks(y)
    ax1.set_yticklabels(y_labels, fontsize=10, weight='bold')
    ax1.set_xlabel('Composição dos Resultados (%)', fontsize=11)
    ax1.set_xlim(0, 102)
    ax1.set_title('Análise de Eficácia por Configuração', fontsize=12, weight='bold', pad=10)
    ax1.grid(axis='x', linestyle=':', alpha=0.5, zorder=0)
    ax1.legend(loc='lower left', fontsize=9)

    # -----------------------------------------------------------------
    # GRÁFICO 2 (Base): Tempo de Execução Horizontal (Anti-viés: iniciando em 0)
    # -----------------------------------------------------------------
    tons_azul = plt.cm.Blues(np.linspace(0.4, 0.7, num_configuracoes))
    bars_time = ax2.barh(y, tempos, color=tons_azul, height=altura_barra, edgecolor='black', linewidth=0.7, zorder=3)
    
    # Texto com o valor bruto do tempo ao lado de cada barra
    for bar in bars_time:
        width = bar.get_width()
        ax2.text(width + (max(tempos) * 0.015), bar.get_y() + bar.get_height()/2., f"{width:.0f}s", 
                 ha='left', va='center', weight='bold', fontsize=8.5, zorder=4)
        
    ax2.set_xlim(0, max(tempos) * 1.35) # Espaço para os grampos laterais de tempo
    
    # Ganchos laterais de comparação de tempo (Par a Par)
    for idx in range(num_configuracoes - 1):
        t_atual = tempos[idx]
        t_proximo = tempos[idx+1]
        d_s = t_proximo - t_atual
        d_pct = (d_s / t_atual) * 100
        
        # Alterna o recuo lateral dos ganchos para evitar sobreposição de linhas
        x_linha_cota = max(tempos) + (max(tempos) * 0.08 * ((idx % 2) + 1))
        
        # Desenha o grampo de medição horizontal
        ax2.plot([t_atual + (max(tempos)*0.02), x_linha_cota, x_linha_cota, t_proximo + (max(tempos)*0.02)], 
                 [idx, idx, idx+1, idx+1], color='gray', linewidth=0.9, linestyle='-', zorder=2)
        
        sinal_t = "+" if d_s > 0 else ""
        texto_tempo = f"Δ: {d_s:.0f}s ({sinal_t}{d_pct:.1f}%)"
        
        ax2.text(x_linha_cota, idx + 0.5, texto_tempo, color='black', weight='bold', fontsize=7.5,
                 ha='center', va='center', zorder=4, rotation=270, # Rotacionado para caber na lateral
                 bbox=dict(facecolor='#fdfdfd', edgecolor='gray', boxstyle='square,pad=0.2', alpha=0.95))

    # Customização do Gráfico 2 (Tempo)
    ax2.set_ylabel('Configurações (IDs)', fontsize=11)
    ax2.set_xlabel('Tempo de Execução (segundos)', fontsize=11)
    ax2.set_title('Análise de Custo Temporal por Configuração', fontsize=12, weight='bold', pad=10)
    ax2.grid(axis='x', linestyle=':', alpha=0.5, zorder=0)

    # Ajustes finais de exportação acadêmica
    plt.tight_layout()
    plt.savefig(nome_arquivo_saida, dpi=300)
    plt.close()
    
    # Tabela gerada invertida para manter a coerência visual (C1 no topo da tabela)
    df_tabela = df.iloc[::-1].reset_index(drop=True)
    print(f"\n=== TABELA DE LEGENDA PARA O ARTIGO (Salvo em {nome_arquivo_saida}) ===")
    print("| ID | Rerank | Filtro API | Lista Filtro | Modelo Chat | Classificador | Embedding |")
    print("|----|--------|------------|--------------|-------------|---------------|-----------|")
    for _, r in df_tabela.iterrows():
        print(f"| {r['ID_Config']} | {r['rerank']} | {r['filtroApi']} | {r['listaFiltroApis']} | {r['modeloChat']} | {r['modeloClassificador']} | {r['embedding']} |")

# Executar a função com o arquivo final
gerar_grafico_horizontal('melhores.csv', 'melhores.png')


=== TABELA DE LEGENDA PARA O ARTIGO (Salvo em melhores.png) ===
| ID | Rerank | Filtro API | Lista Filtro | Modelo Chat | Classificador | Embedding |
|----|--------|------------|--------------|-------------|---------------|-----------|
| C1 | False | True | True | gemma4:e2b | gemma4:e2b | colunas |
| C2 | False | True | True | gemma4:e4b | gemma4:e4b | colunas |
| C3 | False | True | True | qwen2:7b | phi3:3.8b | colunas |
| C4 | False | False | True | gemma4:e2b | gemma4:e2b | colunas |
| C5 | False | False | False | gemma4:e2b | gemma4:e2b | colunas |
| C6 | False | False | True | qwen2:7b | phi3:3.8b | colunas |
| C7 | False | False | True | gemma4:e4b | gemma4:e4b | colunas |
| C8 | False | False | False | gemma4:e4b | gemma4:e4b | colunas |
| C9 | False | False | False | qwen2:7b | phi3:3.8b | colunas |
